In [8]:
def quadratic_model(p, x):
    a, b, c = p
    return a * x**2 + b * x + c

In [15]:
def calculate_x_intercepts(a, b, c):
    import numpy as np
    discriminant = b**2 - 4*a*c
    if discriminant < 0:
        return [0]  # No real roots
    elif discriminant == 0:
        return [-b / (2*a)]  # One real root
    else:
        root1 = (-b + np.sqrt(discriminant)) / (2*a)
        root2 = (-b - np.sqrt(discriminant)) / (2*a)
        return [min(root1,root2)]  # Two real roots, only showing first

In [10]:
def analysis(x,y,sigma_x,sigma_y):
    import pandas as pd
    import numpy as np
    import scipy as sp
    import matplotlib.pyplot as plt
    from scipy.optimize import curve_fit
    from scipy.odr import ODR, Model, RealData
    
    # Create a Model object for the quadratic function
    model = Model(quadratic_model)
    
    # Number of Monte Carlo iterations
    n_iterations = 10000
    
    # Storage for fit parameters from each iteration (3 parameters: a, b, c)
    fit_parameters = np.zeros((n_iterations, 3))
    
    for i in range(n_iterations):
        # Generate simulated data by adding random noise based on the uncertainties
        x_simulated = x + np.random.normal(0, sigma_x)
        y_simulated = y + np.random.normal(0, sigma_y)
        
        # Create a RealData object using the simulated data
        data = RealData(x_simulated, y_simulated, sx=sigma_x, sy=sigma_y)
        
        # Setup ODR with the model and simulated data
        odr = ODR(data, model, beta0=[1.0, 0.0, 0.0])  # Initial guess for a, b, c
        
        # Run the regression
        out = odr.run()
        
        # Store the fit parameters
        fit_parameters[i, :] = out.beta
    
    # Calculate the mean and standard deviation of the fit parameters
    param_means = np.mean(fit_parameters, axis=0)
    param_stds = np.std(fit_parameters, axis=0)
    
    
    #print(f"Parameter estimates (mean ± std): a = {param_means[0]} ± {param_stds[0]}, b = {param_means[1]} ± {param_stds[1]}, c = {param_means[2]} ± {param_stds[2]}")
    
    # Parameters and their uncertainties
    a, b, c = param_means
    sigma_a, sigma_b, sigma_c = param_stds
    
    # Monte Carlo simulation
    n_samples = 10000
    a_samples = np.random.normal(a, sigma_a, n_samples)
    b_samples = np.random.normal(b, sigma_b, n_samples)
    c_samples = np.random.normal(c, sigma_c, n_samples)
    
    x_intercepts = []
    
    for a_sample, b_sample, c_sample in zip(a_samples, b_samples, c_samples):
        x_intercepts.append(calculate_x_intercepts(a_sample, b_sample, c_sample))
    
    # Estimate the uncertainty in the x-intercept
    x_intercept_mean = np.mean(x_intercepts)
    x_intercept_uncertainty = np.std(x_intercepts)
    
    #print(f"Estimated mean in the x-intercept: {x_intercept_mean}")
    #print(f"Estimated uncertainty in the x-intercept: {x_intercept_uncertainty}")
    return(param_means,param_stds,x_intercept_mean,x_intercept_uncertainty)


In [1]:
def gen_cutoff(time,temp,fields,n):
    # Calculates cutoffs for stable temps
    
    index=0
    indices=[]
    for i in np.arange(500,len(temp)):
        if i-index>600:  
            if abs(temp[i]-temp[i-500])>.5 and abs(np.mean(temp[i-500-n:i-500])-np.mean(temp[i-500-2*n:i-500-n]))<.05 or i==len(temp)-1:
                indices.append(i-500)
                #plot_fields(time,fields,i-500-n,cutoff=i-500)
                index=i

    return indices

In [39]:
def field_avg(time,fields,cutoffs,tcomp,n):
    j=0
    means=[]
    index=[]
    temps=[]
    for i in cutoffs:
        #plt.subplots(figsize=(5, 3))
        #plt.plot(time[j:i],fields[j:i])
        index.append((i+j)/2)
        means.append(np.mean(fields[i-n:i])) #Take mean of prev n secs
        temps.append(np.mean(temp[i-n:i])) #changed from i-n to i
        j=i

    #plt.subplots(figsize=(10, 6))
    #plt.scatter(temps,means)
    z=np.arange(min(temps),max(temps),.1)
    fit=np.polyfit(temps,means,1)
    
    #plt.plot(z,fit[0]*z+fit[1], color="lime")
    #plt.title("Field vs Temp for Fluxgate y")
    #plt.xlabel("Temp (C)")
    #plt.ylabel("Field (G)")
    tc_back=[]
    for i in np.arange (9): #doing it for each stable temp
        tc_back.append(fit[0]*temps[i]+fit[1])
        #print("Background at Tc in USBR is", round(tc_back,4))
    return(temps,means,tc_back)

In [18]:
def tc_back():
    return [-0.004656832594633061, -0.00470208934282528, 0.0038095898835512126, -0.00856517436398549, -0.02068007597331315, 0.004741457286249929, 0.005031548379799914, 0.005446858421470433]

In [1]:
def total_back():
    slow=pd.read_csv('nsrgsc_20240613_2_1.csv', header=None)
    begin=10
    #start=1500
    #end=len(slow)-5000
    end=len(slow)
    
    #start=10
    #end=len(slow)
    
    TUSBR = np.float64(slow[2][begin:end])
    TRADS = np.float64(slow[3][begin:end])
    TDSTR = np.float64(slow[4][begin:end])
    TDSBL = np.float64(slow[5][begin:end])
    temps = [TUSBR,TDSTR,TDSBL]
    temp=TUSBR
    
    #Fluxgate readouts, making the upstream ones negative and dividing x,y,z by 4:
    field1 = np.float64(slow[18][begin:end])
    field2 = np.float64(slow[19][begin:end])
    field3 = np.float64(slow[20][begin:end])
    field4 = -np.float64(slow[21][begin:end])
    field5 = np.float64(slow[22][begin:end])
    fieldx = -np.float64(slow[23][begin:end])/4
    fieldy = -np.float64(slow[24][begin:end])/4
    fieldz = -np.float64(slow[25][begin:end])/4
    error = np.float64(slow[6][begin:end])
    index = np.float64(slow[0][begin:end])
    time = np.float64(slow[1][begin:end])
    
    fields=[field1,field2,field3,field4,field5,fieldx,fieldy,fieldz]
    places=["DSTL", "DSTR", "DSBR", "USTL", "DSBL", "USTR", "USBR", "USBL"]

    magnet_list=[0.001451836000000012, 0.0013650080000000009, -0.0011625690000000022, 0.0010884850000000001, 0.0004950519999999997, -0.0009703032499999709, -0.0008884087499999027, -0.0010287609999999892]
    
    #Calculate background for each field probe at each position
    magnet_back=[float(i) for i in magnet_list]
    
    
    n=60 #secs to get data from
    cutoffs=gen_cutoff(time,temp,fields,n)
    back = np.ones((8,9))
    
    for i in np.arange(len(fields)):
        back[i]=field_avg(time,fields[i],cutoffs,-22.5,n)[1] + np.ones(9)*magnet_back[i] #adding ambient background to the additional background from 8T magnet
    
    return(back)
